# Разминка

# A. Раскодируй строку

### Условие

Вася начал изучать кодирование. В этот раз он изобрёл свой шифр, который меняет каждый символ строки по следующему правилу:

- Символы с «a» по «i» отображаются в числа от «1» до «9» соответственно.
- Символы с «j» по «z» отображаются в числа от «10#» до «26#» соответственно.

Например, строка `hello` по этому правилу будет закодирована последовательностью `8512#12#15#`.

Вася научился кодировать строки. А вот с раскодированием у него проблемы. Помогите Васе раскодировать строку!

### Формат ввода

В первой строке дана закодированная последовательность `s` (1 ≤ |s| ≤ 100), состоящая из цифр и символа `#`.

Гарантируется, что последовательность получена кодированием какой-то исходной строки, состоящей из строчных латинских букв.
### Формат вывода

Выведите единственную строку — раскодированную последовательность `s`.


### Решение 1

Время: O(N)

Дополнительная память: O(N)

In [ ]:
def main(text):
    out = []
    pos = 0
    while pos < len(text):
        if pos + 2 < len(text) and text[pos + 2] == '#':
            num = text[pos:pos+2]
            pos += 3
        else:
            num = text[pos]
            pos += 1
        out.append(chr(ord('a') + int(num) - 1))
    return ''.join(out)


if __name__ == "__main__":
    text = input()
    print(main(text))

'abl'

### Решение 2

Время: O(N)

Дополнительная память: O(1)

In [ ]:
def main(text):  # печатает напрямую, без хранения
    pos = 0
    while pos < len(text):
        if pos + 2 < len(text) and text[pos + 2] == '#':
            num = text[pos:pos+2]
            pos += 3
        else:
            num = text[pos]
            pos += 1
        print(chr(ord('a') + int(num) - 1), end='')


if __name__ == "__main__":
    text = input()
    main(text)

## B. Олигополия

### Условие

В игре «Олигополия» N компаний. Для каждой компании известна ее капитализация, для i-й компании она равна ai бурлей. Компания i может поглотить компанию j, если ai строго больше aj, после этого компания j исчезает, а капитал компании i становится равен ai + aj. Поглощения могут происходить в произвольном порядке. Побеждает компания, которая смогла поглотить все остальные.

Вам даны стартовые капиталы компаний. Определите, какие из них, теоретически, могут стать победителями.

### Формат ввода

В первой строке вводится число N, 1 ≤ N ≤ 10^5 — количество компаний.

Во второй строке записано N чисел ai (1 ≤ ai ≤ 10^9) — стартовые капиталы компаний. Числа заданы в порядке неубывания.

### Формат вывода

Выведите N чисел 0 или 1. i-е число должно быть равно 0, если i-я компания не может стать победителем, и 1 — если может.


### Решение 1 - бинарный поиск

Время: O(N log N)

Дополнительная память:


Логично, что если компания смогла поглатить всех предудущих, то следующая за ней компания тоже сможет. Иедя, найдем ту самую первую компания, которая смогла поглотить все предыдущие. Компании меньше, уже не смогут и им поставим 0. Эту компанию можно найти через бинарный поиск.

In [ ]:
def can_win(pos: int, comps: list[int]) -> bool:
    capital = comps[pos]
    for i, current_comp in enumerate(comps):
        if i == pos:
            continue
        if capital > current_comp:
            capital += current_comp
        else:
            return False
    return True


def binary_search(n: int, comps: list[int]) -> list[int]:
    l, r = 0, n
    while l < r:
        mid = (l + r) // 2
        if can_win(mid, comps):
            r = mid
        else:
            l = mid + 1

    # Весьма медленный способ на Python
    # comps = [0] * n
    # for i in range(l, n):
    #     comps[i] = 1

    # print(*comps, sep=' ')
    return ([0] * l + [1] * (n - l))

if __name__ == "__main__":
    n = int(input())
    comps = list(map(int, input().split()))
    print(*binary_search(n, comps), sep=' ')

4
1 1 1 100
0 0 0 1


### Решение 2 - префиксные суммы (авторский вариант)

Время: O(N)

Дополнительная память: O(N)

### Префиксные суммы

Построение массива префиксных сумм:
* Массив можно построить за O(N): prefixsum[i] = prefixsum[i-1] + nums[i]

* Размер массива префексных сумм на 1 элемент больше.

* Может произойти переполнение.


```cpp
long long prefixsum[100];  // Делаем long long
int nums[100];
prefixsum[0] = nums[0];
for (int i = 1; i < n; i++) {
    prefixsum[i] = prefixsum[i-1] + (long long)nums[i];  // Каст перед сложением
    // Или проверка: if (prefixsum[i-1] > LLONG_MAX - nums[i]) error;
}
```

In [ ]:
def solve(n: int, comps: list[int]) -> None:
    if n == 1:
        return [1]

    now_sum = comps[0]
    first_winner = n
    last_loser = 0

    for i in range(1, n - 1):
        now_sum += comps[i]
        if comps[i] > comps[i - 1] and now_sum > comps[i + 1]:
            if first_winner == n:  # Первый победитель
                first_winner = i
        if now_sum <= comps[i + 1]:  # Последняя компания, которая не может поглатить следующих
            last_loser = i

    # Определяем первого победителя
    winner_pos = max(first_winner, last_loser + 1)
    ans = [0] * winner_pos + [1] * (n - winner_pos)
    if comps[n - 1] > comps[n - 2]: # Последнюю компанию мы не обработали случай - 1 1 1 100
        ans[n - 1] = 1
    return ans


if __name__ == "__main__":
    n = int(input())
    comps = list(map(int, input().split()))
    print(*solve(n, comps), sep=' ')

4
1 1 1 10
0 0 0 1


first_winner - говорит о том, что мы можем запустить цепочку поедания - скушать всех до. Мы точно знаем, что начиная с first_winner все предыдущие были бы поглащены.

last_loser - говорит о том, что все после него не будут иметь препятсвий для поглашения.


### Решение 2

Время: O(N)

Дополнительная память: O(N)

In [ ]:
def main(n, comps):
    if n == 1:
        return [1]

    capital = [comps[0]] * n
    for i in range(1, n):
        capital[i] = capital[i-1] + comps[i]

    eat = [True] * (n)  # руппа первых i+1 компаний может съесть ВСЕХ компаний впереди?
    for i in range(n-2, -1, -1):
        eat[i] = eat[i+1] and capital[i] > comps[i+1]
    return [1 if comps[i] > comps[0] and eat[i] else 0 for i in range(n)]


if __name__ == "__main__":
    n = int(input())
    comps = list(map(int, input().split()))
    print(*main(n, comps), sep=' ')

Идея в том, что если компания может съесть всех предыдущих тогда, когда она может поглатить самую слабую. Для этого мы долджны сначала посчитать массив префексных сумм, потом массив групп компаний, которые могут поглатить следующую. И после уже проверить, а может ли каждая из этих компаний съесть слабую, чтобы нарашивать силы.

### Что именно проверяет `eat[i]`
`eat[i]` отвечает на вопрос:

> “Если мы **уже объединили** (съели) компании `0..i` и имеем капитал `capital[i]`, сможем ли мы дальше по очереди съесть `i+1, i+2, ..., n-1`?”

Поэтому условие “можно съесть следующую” записывается как:

- **группа** с капиталом `capital[i]` может съесть следующую компанию `comps[i+1]`, если `capital[i] > comps[i+1]`.


Это не утверждение “компания с капиталом 3 может съесть 3”; это утверждение “компания, которая уже стала капиталом 6 (после поглощений), может съесть 3”.
